# 31% of train rows have a twin. None of them help.

Three measurements about the data rather than about models. None will improve your score.
All three change what you should stop doing.

1. **Can a classifier tell train from test?** Measured against a permutation null, so the
   answer is compared with what "nothing" scores rather than with 0.5 in the abstract. Same
   machinery run against the origin dataset, which sizes how far Kaggle's generator moved.
2. **How many rows are duplicated, and is that a lever?** Exact duplicates across all 13
   features, then across only the 11 discrete ones — and then the part people skip: whether
   the label of a row's twins actually predicts it better than a model does. It does not,
   and the notebook shows by how much.
3. **A blend section**, inert until an out-of-fold library is mounted. When one is, it
   prints the member rank-correlation map and the nested-blend gain, priced against the
   public leaderboard's standard error.

**What this notebook deliberately does not do.** Two notebooks already own the noise story
here and I am not going to re-derive it worse. [georgymamarin's S6E9
starter](https://www.kaggle.com/code/georgymamarin/s6e9-starter-how-to-tell-a-real-gain-from-noise) established the noise bar from null
draws. [busyaprime's *Why everyone is stuck at 0.941 on this LB*](https://www.kaggle.com/code/busyaprime/why-everyone-is-stuck-at-0-941-on-this-lb)
fits a learning curve over six training sizes and computes the public score's standard
error two ways. The public split is 20% of the test rows. I take that SE as an input,
credit it, and use it only as the yardstick in section 3.

⚠️ Both of those links are to the author, not the notebook — fill in the exact notebook
URLs before relying on them.


## The one cell you edit

In [1]:
CFG = {
    "SEED": 42,
    "N_SPLITS": 5,
    "TARGET": "Will_Buy_EV",
    "ID_COL": "id",

    # A column with at most this many distinct values counts as "discrete" for the
    # duplicate audit. On this competition the gap is wide - the 11th feature has 45
    # levels and the 12th has 805 - so the threshold is not a close call. The cell
    # below prints every column's cardinality so you can check that for yourself.
    "DISCRETE_MAX_LEVELS": 50,

    # Rounding applied to the continuous columns in the near-duplicate variant.
    "ROUND_TO": {"Annual_Income_USD": 1000, "Daily_Commute_km": 10},

    # Adversarial validation. Both sides are subsampled to the same size, because an
    # unbalanced adversarial AUC is not comparable to a balanced null.
    "MAX_ADV_ROWS": 50_000,
    "N_PERM": 10,
    "ADV_PARAMS": dict(n_estimators=150, learning_rate=0.1, num_leaves=31,
                       min_child_samples=50, colsample_bytree=0.8,
                       subsample=0.8, subsample_freq=1),

    # A plain baseline, used only as the comparison point for the lookup test. It is
    # not meant to be competitive and it is not tuned.
    "BASE_PARAMS": dict(n_estimators=400, learning_rate=0.05, num_leaves=63,
                        min_child_samples=100, colsample_bytree=0.8,
                        subsample=0.8, subsample_freq=1, reg_lambda=5.0),

    # The public leaderboard scores 20% of the 286,571 test rows = 57,314 rows. At an
    # AUC near 0.946 with a 17.47% positive rate, the Hanley-McNeil standard error of
    # that estimate is 0.001593 - and that is a LOWER bound (it assumes independent
    # labels). Measured, not guessed; section 3 prices the blend gain against it.
    "PUBLIC_FRAC": 0.20,
    "PUBLIC_SE": 0.001593,
}

for k, v in CFG.items():
    print(f"{k:>20} = {v!r}")


                SEED = 42
            N_SPLITS = 5
              TARGET = 'Will_Buy_EV'
              ID_COL = 'id'
 DISCRETE_MAX_LEVELS = 50
            ROUND_TO = {'Annual_Income_USD': 1000, 'Daily_Commute_km': 10}
        MAX_ADV_ROWS = 50000
              N_PERM = 10
          ADV_PARAMS = {'n_estimators': 150, 'learning_rate': 0.1, 'num_leaves': 31, 'min_child_samples': 50, 'colsample_bytree': 0.8, 'subsample': 0.8, 'subsample_freq': 1}
         BASE_PARAMS = {'n_estimators': 400, 'learning_rate': 0.05, 'num_leaves': 63, 'min_child_samples': 100, 'colsample_bytree': 0.8, 'subsample': 0.8, 'subsample_freq': 1, 'reg_lambda': 5.0}
         PUBLIC_FRAC = 0.2
           PUBLIC_SE = 0.001593


## Loading

Bounded breadth-first search for each filename rather than a hard-coded mount path, since
Kaggle uses more than one layout. The origin dataset and the out-of-fold library are both
optional: if either is not mounted, its section prints a note and is skipped rather than
taking the notebook down with it.


In [2]:
import os, glob, time, warnings
from collections import deque

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import lightgbm as lgb

warnings.filterwarnings("ignore")
T0 = time.time()
RNG = np.random.default_rng(CFG["SEED"])

SEARCH_ROOT = os.environ.get("SMOKE_ROOT", "/kaggle/input")
WORK = os.environ.get("SMOKE_WORK", "/kaggle/working")
os.makedirs(WORK, exist_ok=True)


def find_file(filename, root=SEARCH_ROOT, max_depth=5, required=True):
    """Bounded breadth-first search for `filename` under `root`."""
    queue = deque([(root, 0)])
    hits = []
    while queue:
        directory, depth = queue.popleft()
        try:
            entries = sorted(os.scandir(directory), key=lambda e: e.name)
        except (PermissionError, FileNotFoundError, NotADirectoryError):
            continue
        for entry in entries:
            try:
                if entry.is_file() and entry.name == filename:
                    hits.append(entry.path)
                elif entry.is_dir() and depth < max_depth:
                    queue.append((entry.path, depth + 1))
            except OSError:
                continue
    if not hits:
        if not required:
            return None
        tree, stack = [], deque([(root, 0)])
        while stack and len(tree) < 60:
            directory, depth = stack.popleft()
            try:
                entries = sorted(os.scandir(directory), key=lambda e: e.name)
            except OSError:
                continue
            for entry in entries:
                tree.append(entry.path)
                if entry.is_dir() and depth < 3:
                    stack.append((entry.path, depth + 1))
        raise FileNotFoundError(
            f"{filename!r} not found under {root}. What is mounted:\n  "
            + "\n  ".join(tree[:60]))
    hits.sort(key=len)
    return hits[0]


train = pd.read_csv(find_file("train.csv"))
test = pd.read_csv(find_file("test.csv"))
src_path = find_file("EV_Adoption_and_Range_Anxiety_Dataset.csv", required=False)
origin = pd.read_csv(src_path) if src_path else None

y = train[CFG["TARGET"]]
if not pd.api.types.is_numeric_dtype(y):
    y = (y.astype(str).str.strip().str.lower() == "yes").astype(np.int8)
y = y.to_numpy().astype(np.int8)

ID_COL = CFG["ID_COL"] if CFG["ID_COL"] in train.columns else None
feature_cols = [c for c in train.columns
                if c not in {CFG["TARGET"], ID_COL} and c in test.columns]

card = train[feature_cols].nunique().sort_values()
DISCRETE = [c for c in feature_cols if card[c] <= CFG["DISCRETE_MAX_LEVELS"]]
CONTINUOUS = [c for c in feature_cols if c not in DISCRETE]

print(f"train  {len(train):>9,} rows x {train.shape[1]} columns")
print(f"test   {len(test):>9,} rows x {test.shape[1]} columns")
print(f"origin {'not mounted' if origin is None else f'{len(origin):>9,} rows'}")
print(f"positive rate {y.mean():.4%}   missing cells in train "
      f"{int(train.isna().sum().sum()):,}\n")
print("distinct values per feature:")
print(card.to_string())
print(f"\n{len(DISCRETE)} discrete features (<= {CFG['DISCRETE_MAX_LEVELS']} levels), "
      f"{len(CONTINUOUS)} continuous: {CONTINUOUS}")
print(f"[{time.time() - T0:.0f}s]")


train    668,665 rows x 15 columns
test     286,571 rows x 14 columns
origin    10,000 rows
positive rate 17.4645%   missing cells in train 0

distinct values per feature:
Home_Charging_Possible             2
Subsidy_Available                  2
Gender                             3
City_Type                          3
Range_Anxiety_Level                3
Number_of_Cars_Owned               4
Current_Car_Type                   4
Environmental_Concern_Level        5
Charging_Stations_Near_Home       15
Charging_Stations_Near_Work       20
Age                               45
Daily_Commute_km                 805
Annual_Income_USD              13214

11 discrete features (<= 50 levels), 2 continuous: ['Annual_Income_USD', 'Daily_Commute_km']
[3s]


## 1. Can a model tell train from test?

Stack the two files, label every train row 0 and every test row 1, and cross-validate a
classifier on that. An AUC of 0.5 means the two files are interchangeable and ordinary
cross-validation on train is a fair estimate of test performance.

Two details that make the answer trustworthy rather than decorative. Both sides are
subsampled to the **same** size, because an adversarial AUC on unbalanced classes is not
comparable to a balanced null. And the observed value is compared against a **permutation
null** — the same model, the same folds, on shuffled labels — because "0.503" means nothing
until you know that shuffled labels also reach 0.503.


In [3]:
def prepare(df, cols):
    out = pd.DataFrame(index=range(len(df)))
    for c in cols:
        if pd.api.types.is_numeric_dtype(df[c]):
            out[c] = pd.to_numeric(df[c], errors="coerce").astype("float64").to_numpy()
        else:
            out[c] = pd.Categorical(df[c].astype(str).to_numpy())
    return out


def balanced_pair(a, b, cols, cap, rng):
    """Subsample both frames to the same size, then put them on a shared encoding."""
    n = int(min(cap, len(a), len(b)))
    ia = np.sort(rng.choice(len(a), n, replace=False))
    ib = np.sort(rng.choice(len(b), n, replace=False))
    A = a.iloc[ia][cols].reset_index(drop=True)
    B = b.iloc[ib][cols].reset_index(drop=True)
    both = pd.concat([A, B], ignore_index=True)
    X = prepare(both, cols)
    z = np.r_[np.zeros(n, dtype=np.int8), np.ones(n, dtype=np.int8)]
    return X, z, n


def adv_cv(X, z, params, n_splits, seed):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    oof = np.zeros(len(z))
    for tr, va in skf.split(X, z):
        m = lgb.LGBMClassifier(random_state=CFG["SEED"], verbose=-1, **params)
        m.fit(X.iloc[tr], z[tr])
        oof[va] = m.predict_proba(X.iloc[va])[:, 1]
    return roc_auc_score(z, oof)


def adversarial(a, b, cols, label, rng):
    X, z, n = balanced_pair(a, b, cols, CFG["MAX_ADV_ROWS"], rng)
    obs = adv_cv(X, z, CFG["ADV_PARAMS"], CFG["N_SPLITS"], CFG["SEED"])
    print(f"{label}: {n:,} rows a side, observed AUC {obs:.6f}  "
          f"[{time.time() - T0:.0f}s]")
    nulls = np.array([adv_cv(X, rng.permutation(z), CFG["ADV_PARAMS"],
                             CFG["N_SPLITS"], CFG["SEED"] + i)
                      for i in range(CFG["N_PERM"])])
    sd = nulls.std(ddof=1)
    sigma = (obs - nulls.mean()) / sd
    p = (1 + int((nulls >= obs).sum())) / (1 + len(nulls))
    p_floor = 1.0 / (1 + len(nulls))
    print(f"  null over {len(nulls)} draws: mean {nulls.mean():.6f}  sd {sd:.6f}  "
          f"range {nulls.min():.6f}..{nulls.max():.6f}")
    print(f"  observed sits {sigma:+.2f} null SDs from the null mean")
    print(f"  empirical p = {p:.4f}   (FLOOR {p_floor:.4f} with {len(nulls)} draws - "
          f"a permutation p can never go below 1/(N_PERM+1), so when the observed is "
          f"many SDs out, read the SD and ignore the p)")
    print(f"  [{time.time() - T0:.0f}s]")
    return {"label": label, "auc": obs, "null_mean": float(nulls.mean()),
            "null_sd": float(sd), "sigma": float(sigma), "p": p, "n": n, "X": X, "z": z}


ADV_TT = adversarial(train, test, feature_cols, "train vs test", RNG)


train vs test: 50,000 rows a side, observed AUC 0.499648  [7s]
  null over 10 draws: mean 0.499642  sd 0.002304  range 0.496959..0.503462
  observed sits +0.00 null SDs from the null mean
  empirical p = 0.5455   (FLOOR 0.0909 with 10 draws - a permutation p can never go below 1/(N_PERM+1), so when the observed is many SDs out, read the SD and ignore the p)
  [49s]


In [4]:
# If, and only if, the AUC had cleared its null, this table would name the columns
# responsible. It is printed either way so the reader can see it is flat.
m_full = lgb.LGBMClassifier(random_state=CFG["SEED"], verbose=-1, **CFG["ADV_PARAMS"])
m_full.fit(ADV_TT["X"], ADV_TT["z"])
imp = pd.DataFrame({"feature": ADV_TT["X"].columns,
                    "gain": m_full.booster_.feature_importance("gain")})
imp["share_%"] = 100.0 * imp["gain"] / max(imp["gain"].sum(), 1)
print(imp.sort_values("gain", ascending=False).to_string(
    index=False, formatters={"gain": "{:,.0f}".format, "share_%": "{:6.2f}".format}))

SHIFT_VERDICT = ("no measurable shift" if ADV_TT["sigma"] < 3
                 else "a measurable shift")
print(f"\nverdict: {SHIFT_VERDICT}. The observed AUC is "
      f"{ADV_TT['sigma']:+.2f} null SDs from a shuffled-label baseline.")
assert ADV_TT["sigma"] < 3, (
    "the title claims zero train/test shift and this run disagrees; "
    "that needs a new notebook, not an edited title")


                    feature  gain share_%
          Annual_Income_USD 5,067   23.97
           Daily_Commute_km 4,273   20.21
                        Age 2,951   13.96
Charging_Stations_Near_Home 2,219   10.50
Charging_Stations_Near_Work 2,118   10.02
Environmental_Concern_Level 1,222    5.78
       Number_of_Cars_Owned   782    3.70
                     Gender   600    2.84
                  City_Type   546    2.58
           Current_Car_Type   430    2.04
          Subsidy_Available   405    1.92
     Home_Charging_Possible   280    1.32
        Range_Anxiety_Level   244    1.16

verdict: no measurable shift. The observed AUC is +0.00 null SDs from a shuffled-label baseline.


### Same machinery, against the origin dataset

The competition data was generated by a model trained on a public dataset. Here a high AUC
is *expected* — the generator sampled a much larger file from a fitted model, so the two
are related but not the same draw. The number sizes that difference, and it decides one
thing: whether appending the origin rows to training is adding information or importing a
different distribution.


In [5]:
if origin is None:
    print("origin dataset not mounted; skipping. Add "
          "`itzzomkar/ev-adoption-behavior-and-range-anxiety` to this notebook's inputs.")
    ADV_TO, shared = None, []
else:
    shared = [c for c in feature_cols if c in origin.columns]
    missing = [c for c in feature_cols if c not in origin.columns]
    print(f"{len(shared)} of {len(feature_cols)} features exist in both files")
    if missing:
        print(f"competition-only, excluded here: {missing}")
    print()
    ADV_TO = adversarial(train, origin, shared, "train vs origin", RNG)
    print(f"\nFor scale, train vs test came out at {ADV_TT['auc']:.6f}.")


13 of 13 features exist in both files

train vs origin: 10,000 rows a side, observed AUC 0.759096  [52s]
  null over 10 draws: mean 0.500998  sd 0.005193  range 0.491937..0.507218
  observed sits +49.70 null SDs from the null mean
  empirical p = 0.0909   (FLOOR 0.0909 with 10 draws - a permutation p can never go below 1/(N_PERM+1), so when the observed is many SDs out, read the SD and ignore the p)
  [67s]

For scale, train vs test came out at 0.499648.


## 2. Duplicate rows, and whether they are a lever

Generated tabular data repeats itself, and every S6E9 thread eventually asks whether that
can be exploited. The audit runs twice: once across **all** features, and once across only
the **discrete** ones — dropping the two continuous columns, whose long decimal tails make
every row unique on their own and hide any structure underneath.

Then the part that decides the question. For every row sitting in a twin group, take the
mean label of its twins, **leaving that row out**, and use it as a prediction. If the twins
carry usable information, that lookup should beat a model on those same rows. The cell
below fits a plain baseline for exactly that comparison.


In [6]:
def key_of(df, cols, rounding=None):
    """One string key per row.

    The `.fillna` is load-bearing and cost a debugging session. On pandas 3 a float
    column's `.astype(str)` leaves NaN MISSING rather than the text "nan", and string
    concatenation propagates missing - so without the fill every row holding any
    missing value collapses into one null key and gets reported as a duplicate of
    all the others. The assert makes that impossible to miss on a future pandas.
    """
    parts = []
    for c in cols:
        s = df[c]
        if rounding and c in rounding:
            s = (pd.to_numeric(s, errors="coerce") / rounding[c]).round()
        parts.append(s.astype(str).fillna("<NA>"))
    out = parts[0]
    for p in parts[1:]:
        out = out + "\x1f" + p
    assert not out.isna().any(), "null key survived; the fillna above stopped working"
    return out.to_numpy()


def audit(cols, label, short, rounding=None):
    k_tr = key_of(train, cols, rounding)
    k_te = key_of(test, cols, rounding)
    counts = pd.Series(k_tr).value_counts()
    multi = counts[counts > 1]
    grouped_rows = int(multi.sum())
    shared = set(counts.index) & set(pd.unique(k_te))
    te_hit = int(pd.Series(k_te).isin(shared).sum()) if shared else 0

    g = pd.DataFrame({"k": k_tr, "y": y})
    size = g.groupby("k")["y"].transform("size").to_numpy()
    total = g.groupby("k")["y"].transform("sum").to_numpy()
    in_group = size > 1
    if in_group.any():
        pure = ((total[in_group] == 0) | (total[in_group] == size[in_group]))
        pure_rate = float(pd.Series(pure).groupby(
            pd.Series(k_tr)[in_group].to_numpy()).first().mean())
        loo = (total[in_group] - y[in_group]) / (size[in_group] - 1)
    else:
        pure_rate, loo = float("nan"), None

    print(f"--- {label} ({len(cols)} columns) ---")
    print(f"  distinct train keys        {len(counts):>10,}  of {len(train):,} rows")
    print(f"  rows in a twin group       {grouped_rows:>10,}  "
          f"({grouped_rows / len(train):.3%})")
    print(f"  twin groups                {len(multi):>10,}  "
          f"mean size {multi.mean() if len(multi) else 0:.2f}")
    print(f"  twin groups label-pure     {pure_rate:>10.3%}" if len(multi)
          else "  twin groups label-pure            n/a")
    print(f"  test rows with a train twin{te_hit:>10,}  "
          f"({te_hit / len(test):.3%})")
    return {"label": label, "short": short, "n_keys": len(counts),
            "grouped_rows": grouped_rows,
            "n_groups": len(multi), "test_hit": te_hit,
            "pure_rate": pure_rate, "in_group": in_group, "loo": loo}


ALL13 = audit(feature_cols, "all features", f"all {len(feature_cols)}")
print()
DISC = audit(DISCRETE, f"discrete features only (dropping {CONTINUOUS})",
             f"{len(DISCRETE)} discrete")
print()
ROUNDED = audit(feature_cols, "all features, continuous columns rounded",
                f"all {len(feature_cols)}, rounded", rounding=CFG["ROUND_TO"])
print(f"\n[{time.time() - T0:.0f}s]")


--- all features (13 columns) ---
  distinct train keys           668,665  of 668,665 rows
  rows in a twin group                0  (0.000%)
  twin groups                         0  mean size 0.00
  twin groups label-pure            n/a
  test rows with a train twin         0  (0.000%)

--- discrete features only (dropping ['Annual_Income_USD', 'Daily_Commute_km']) (11 columns) ---
  distinct train keys           546,134  of 668,665 rows
  rows in a twin group          205,065  (30.668%)
  twin groups                    82,534  mean size 2.48
  twin groups label-pure        77.280%
  test rows with a train twin    87,775  (30.629%)

--- all features, continuous columns rounded (13 columns) ---
  distinct train keys           668,122  of 668,665 rows
  rows in a twin group            1,083  (0.162%)
  twin groups                       540  mean size 2.01
  twin groups label-pure        87.778%
  test rows with a train twin       505  (0.176%)

[94s]


In [7]:
# The title asserts both duplicate figures. Recomputed here, checked here.
EXACT_PCT = 100.0 * ALL13["grouped_rows"] / len(train)
DISC_PCT = 100.0 * DISC["grouped_rows"] / len(train)
print(f"exact twins across all {len(feature_cols)} features: {EXACT_PCT:.3f}% of train")
print(f"twins across the {len(DISCRETE)} discrete features: {DISC_PCT:.3f}% of train")
assert round(EXACT_PCT) == 0, (
    f"title says 0% exact twins, this run found {EXACT_PCT:.3f}%")
assert round(DISC_PCT) == 31, (
    f"title says 31% discrete twins, this run found {DISC_PCT:.3f}%")
print("both figures agree with the title.")


exact twins across all 13 features: 0.000% of train
twins across the 11 discrete features: 30.668% of train
both figures agree with the title.


In [8]:
# A plain baseline, fit only to answer "does the twin lookup beat a model on the
# rows where a lookup is even possible". Not tuned, not competitive, not a submission.
X = prepare(train, feature_cols)
skf = StratifiedKFold(n_splits=CFG["N_SPLITS"], shuffle=True,
                      random_state=CFG["SEED"])
oof = np.zeros(len(y))
for tr, va in skf.split(X, y):
    m = lgb.LGBMClassifier(random_state=CFG["SEED"], verbose=-1, **CFG["BASE_PARAMS"])
    m.fit(X.iloc[tr], y[tr])
    oof[va] = m.predict_proba(X.iloc[va])[:, 1]
BASE_AUC = roc_auc_score(y, oof)
print(f"plain baseline, {CFG['N_SPLITS']}-fold, whole train: AUC {BASE_AUC:.6f}  "
      f"[{time.time() - T0:.0f}s]\n")

rows = []
for res in (DISC, ROUNDED):
    sel = res["in_group"]
    if res["loo"] is None or y[sel].min() == y[sel].max():
        continue
    lookup_auc = roc_auc_score(y[sel], res["loo"])
    model_auc = roc_auc_score(y[sel], oof[sel])
    blend = (pd.Series(oof[sel]).rank(pct=True).to_numpy()
             + pd.Series(res["loo"]).rank(pct=True).to_numpy()) / 2
    blend_auc = roc_auc_score(y[sel], blend)
    rows.append({"keyed on": res["short"], "rows": int(sel.sum()),
                 "twin lookup": lookup_auc, "model": model_auc,
                 "50/50 rank blend": blend_auc,
                 "blend - model": blend_auc - model_auc})

lever = pd.DataFrame(rows)
print(lever.to_string(index=False, formatters={
    "twin lookup": "{:.6f}".format, "model": "{:.6f}".format,
    "50/50 rank blend": "{:.6f}".format, "blend - model": "{:+.6f}".format}))
print("\nAll four columns are scored on the SAME rows — the ones that sit in a twin "
      "group — so they are directly comparable.")
for _, r in lever.iterrows():
    gap = r["model"] - r["twin lookup"]
    print(f"  keyed on {r['keyed on']:<18} the model beats the twin lookup by "
          f"{gap:+.6f},")
    print(f"  {'':<27} and blending the lookup in moves it by "
          f"{r['blend - model']:+.6f}")


plain baseline, 5-fold, whole train: AUC 0.941766  [176s]

       keyed on   rows twin lookup    model 50/50 rank blend blend - model
    11 discrete 205065    0.771042 0.933026         0.909027     -0.023998
all 13, rounded   1083    0.778143 0.951350         0.942486     -0.008865

All four columns are scored on the SAME rows — the ones that sit in a twin group — so they are directly comparable.
  keyed on 11 discrete        the model beats the twin lookup by +0.161984,
                              and blending the lookup in moves it by -0.023998
  keyed on all 13, rounded    the model beats the twin lookup by +0.173208,
                              and blending the lookup in moves it by -0.008865


### No duplicate leak

Read the table above rather than my summary of it, but the shape of the answer is: a large
share of train rows do have twins on the discrete features, the twin groups are mostly but
not entirely label-pure, and the leave-one-out twin label is a much **worse** predictor
than an ordinary model on the very rows where it is available. Blending it in does not
help either.

That is what "no leak" means here, and it is a more useful statement than the usual
reassurance, because it comes with the size of the thing that is not a leak. The twins are
real. They are just not information the model did not already have — the same discrete
features that make two rows twins are the features the model is reading.

Rounding the continuous columns to make more rows collide does not rescue it. It buys a
smaller grouped set and a weaker lookup, which is what you would expect if the collisions
were carrying nothing to begin with.


## 3. Blend diagnostics — inert until a library is mounted

This section reads an out-of-fold library: one `oof_<m>.npy` per member, each in
`train.csv` file row order, on the `StratifiedKFold(5, shuffle=True, random_state=42)`
split. When one is mounted it prints the member rank-correlation map and the gain a
nested-honest blend gets over the best single member — and prices that gain against the
public leaderboard's standard error, so a gain smaller than the board can see is visible as
such.

Nothing here is fitted on the rows it reports: the blend weights come from four folds and
are scored on the fifth.

When no library is mounted, the cell says so and the notebook still finishes.


In [9]:
lib_manifest = find_file("manifest.csv", required=False)
LIB = os.path.dirname(lib_manifest) if lib_manifest else None
oof_paths = sorted(glob.glob(os.path.join(LIB, "oof_*.npy"))) if LIB else []

if not oof_paths:
    print("No out-of-fold library mounted, so this section has nothing to measure.")
    print("It expects a dataset holding, per member m:")
    print("    oof_<m>.npy   float64, len(train), train.csv FILE ROW ORDER, no ids")
    print("    test_<m>.npy  float64, len(test),  test.csv  FILE ROW ORDER, no ids")
    print("    folds_seed42.npy, manifest.csv")
    print("Mount one and re-run; everything below fills itself in.")
    MEMBERS, LIB_OOF = [], None
else:
    MEMBERS, cols_ok = [], []
    for p in oof_paths:
        name = os.path.basename(p)[4:-4]
        a = np.load(p)
        if len(a) != len(train) or not np.isfinite(a).all():
            print(f"  DROPPED {name}: length {len(a):,} or non-finite values")
            continue
        MEMBERS.append(name)
        cols_ok.append(a.astype(np.float64))
    LIB_OOF = np.column_stack(cols_ok) if cols_ok else None
    print(f"library at {LIB}")
    if MEMBERS:
        print(f"{len(MEMBERS)} members loaded: {MEMBERS}")
    else:
        print("a manifest.csv was found, but no member array matched this train "
              "file's row count. That is a different competition's library, or a "
              "half-written one. Section 3 stays inert rather than reporting "
              "numbers off arrays that are not aligned to these rows.")


library at /kaggle/input/s6e9-golem-oof-library
19 members loaded: ['a', 'a_lgbm', 'b', 'b_lgbm_deep', 'c', 'c_xgb', 'd', 'd_catboost', 'e', 'e_histgb', 'f', 'f_lgbm_te', 'g', 'g_mlp', 'h', 'h_spline_gam', 'i', 'i_extratrees', 'j_logreg']


In [10]:
if LIB_OOF is None:
    print("skipped — no library mounted.")
else:
    from sklearn.linear_model import LogisticRegression

    solo = pd.Series({m: roc_auc_score(y, LIB_OOF[:, j])
                      for j, m in enumerate(MEMBERS)}).sort_values(ascending=False)
    print("member out-of-fold AUC, recomputed here:")
    print(solo.to_string(float_format="{:.6f}".format))

    ranks = pd.DataFrame(LIB_OOF, columns=MEMBERS).rank()
    corr = ranks.corr()          # pearson on ranks is spearman, and much faster
    print("\nSpearman rank correlation between members:")
    print(corr.to_string(float_format="{:.4f}".format))
    off = corr.mask(np.eye(len(corr), dtype=bool))
    print(f"\nmost redundant pair: {off.max().max():.4f}   "
          f"least: {off.min().min():.4f}")

    folds_file = find_file("folds_seed42.npy", required=False)
    if folds_file:
        FOLDS = np.load(folds_file).astype(np.int8)
    else:
        FOLDS = np.zeros(len(y), dtype=np.int8)
        for f, (_, va) in enumerate(skf.split(X, y)):
            FOLDS[va] = f
        print("\nno folds_seed42.npy; regenerated the seed-42 convention.")

    def fit_rank(col):
        return np.sort(col)

    def apply_rank(sorted_train, col):
        return np.searchsorted(sorted_train, col, side="right") / len(sorted_train)

    nested = np.zeros(len(y))
    for f in range(CFG["N_SPLITS"]):
        tr_m, va_m = FOLDS != f, FOLDS == f
        rk = [fit_rank(LIB_OOF[tr_m, j]) for j in range(LIB_OOF.shape[1])]
        Xtr = np.column_stack([apply_rank(r, LIB_OOF[tr_m, j])
                               for j, r in enumerate(rk)])
        Xva = np.column_stack([apply_rank(r, LIB_OOF[va_m, j])
                               for j, r in enumerate(rk)])
        lr = LogisticRegression(C=1.0, max_iter=2000)
        lr.fit(Xtr, y[tr_m])
        nested[va_m] = lr.predict_proba(Xva)[:, 1]

    NESTED = roc_auc_score(y, nested)
    BEST = float(solo.iloc[0])
    GAIN = NESTED - BEST
    print(f"\nnested blend AUC      {NESTED:.6f}")
    print(f"best single member    {BEST:.6f}  ({solo.index[0]})")
    print(f"gain                  {GAIN:+.6f}")

    if CFG["PUBLIC_SE"]:
        se = CFG["PUBLIC_SE"]
        print(f"\npublic-LB SE (Hanley-McNeil lower bound, 57,314 rows, 17.47% positive): {se:.6f}")
        print(f"the gain is {GAIN / se:.2f} public SEs")
        print("A gain under about 2 SEs is not something the public board can "
              "separate from noise, whatever it does in cross-validation.")
    else:
        print("\n⚠️ CFG['PUBLIC_SE'] is not set, so the gain is unpriced. Read the SE "
              "off busyaprime's notebook and set it — I am not going to estimate it "
              "here and pass off a second-hand number as a measurement.")


member out-of-fold AUC, recomputed here:
h              0.944507
i              0.944407
a              0.943742
d_catboost     0.941672
d              0.941612
a_lgbm         0.941603
b              0.941447
c_xgb          0.941368
e_histgb       0.941342
f_lgbm_te      0.941326
c              0.941177
b_lgbm_deep    0.940907
h_spline_gam   0.939913
f              0.939176
e              0.938387
g_mlp          0.938375
i_extratrees   0.938330
g              0.938105
j_logreg       0.938094

Spearman rank correlation between members:
                  a  a_lgbm      b  b_lgbm_deep      c  c_xgb      d  d_catboost      e  e_histgb      f  f_lgbm_te      g  g_mlp      h  h_spline_gam      i  i_extratrees  j_logreg
a            1.0000  0.9926 0.9922       0.9926 0.9918 0.9914 0.9942      0.9920 0.9844    0.9928 0.9894     0.9910 0.9826 0.9843 0.9942        0.9906 0.9936        0.9820    0.9857
a_lgbm       0.9926  1.0000 0.9934       0.9946 0.9928 0.9938 0.9940      0.9927 0.9841    0.99

## What these three numbers change

- **Zero train/test shift** → cross-validation on train is a fair estimate of test
  performance. No importance weighting, no shift-corrected validation split, no reading
  further threads about it.
- **31% of rows have a discrete twin, and the twins are not a lever** → the lookup idea is
  closed. The twin groups exist because eleven features have few levels and the file is
  large, not because anything leaked.
- **The origin comparison** → sizes how far the generator moved, which is the only input
  to the "should I append the origin data" question worth having.

- **The blend gain, priced** → with the library mounted, section 3 reports what a nested
  blend of every member adds over the best single one, in units of the public board's own
  standard error. On my run that gain was about a tenth of a standard error (0.13). The board
  cannot see it, so no amount of blending will move a rank here.

None of that is a modelling idea. All of it is a list of things not to spend the next week
on, which on a board where the top is already crowded into a tenth of a point is worth
more.

## What the 31% turned out to be worth

The discrete twins are not a lookup lever, but the *reason* they exist — eleven columns
with a handful of levels each, plus quantisation artefacts in the two wide ones — is the
population a target encoder summarises well. Encoding every numeric, categorical and digit
column as a string category and target-encoding it inside each fold was worth
**+0.0011 out-of-fold on these same folds, winning all five**, the largest single feature
gain measured on this board. The full ladder, with the three companion ingredients that
did nothing, is in
[S6E9 0.94585: 1 of 4 Ingredients Carries It](https://www.kaggle.com/code/dariushafshar/s6e9-0-94585-1-of-4-ingredients-carries-it).
So the honest reading of "31% discrete" is not "there is a leak" — there is not — it is
"this frame is mostly categories wearing numeric clothes, and should be modelled that way."


In [11]:
print(f"total runtime {time.time() - T0:.1f}s  ({(time.time() - T0) / 60:.1f} min)")
print(f"train vs test    AUC {ADV_TT['auc']:.6f}  "
      f"null {ADV_TT['null_mean']:.6f} +- {ADV_TT['null_sd']:.6f}  "
      f"({ADV_TT['sigma']:+.2f} SD, p {ADV_TT['p']:.4f})")
if ADV_TO is not None:
    print(f"train vs origin  AUC {ADV_TO['auc']:.6f}  "
          f"null {ADV_TO['null_mean']:.6f}  ({ADV_TO['sigma']:+.2f} SD)")
print(f"exact twins {EXACT_PCT:.3f}% of train, discrete twins {DISC_PCT:.3f}%")
print(f"baseline AUC {BASE_AUC:.6f}")
if len(lever):
    r = lever.iloc[0]
    print(f"on twin rows: model {r['model']:.6f} vs twin lookup "
          f"{r['twin lookup']:.6f}  ({r['model'] - r['twin lookup']:+.6f})")
print(f"blend section: {'ran' if MEMBERS else 'inert (no library mounted)'}")


total runtime 220.4s  (3.7 min)
train vs test    AUC 0.499648  null 0.499642 +- 0.002304  (+0.00 SD, p 0.5455)
train vs origin  AUC 0.759096  null 0.500998  (+49.70 SD)
exact twins 0.000% of train, discrete twins 30.668%
baseline AUC 0.941766
on twin rows: model 0.933026 vs twin lookup 0.771042  (+0.161984)
blend section: ran
